# Notebook 3: PPO Fine-Tuning

## 3.0 Preamble

This notebook fine-tunes the behavior-cloning (BC) policy from Notebook 02 using Proximal Policy Optimization (PPO) through live interaction with the CARLA simulator. PPO refines the imitation-learned policy by maximizing a shaped reward signal that captures safe, comfortable, and efficient driving.

PPO maximizes the clipped surrogate objective

$$\mathcal{L}^{\text{CLIP}}(\theta) = \mathbb{E}_t\!\left[\min\!\left(\rho_t(\theta)\,\hat{A}_t,\;\operatorname{clip}\!\left(\rho_t(\theta),\,1-\varepsilon,\,1+\varepsilon\right)\hat{A}_t\right)\right]$$

where $\rho_t(\theta) = \pi_\theta(a_t|s_t)\,/\,\pi_{\theta_{\text{old}}}(a_t|s_t)$ is the probability ratio, $\hat{A}_t$ is the Generalized Advantage Estimate (GAE), and $\varepsilon = 0.2$ prevents destructively large policy updates.

The training introduces three distinct driving style variants -- **chill**, **standard**, and **hurry** -- each governed by a different reward-shaping weight vector $(w_j, w_s, w_\delta, w_l)$. Each style produces its own checkpoint `ppo_{style}_{sensor_suite}_best.pt`.

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd

# Ensure project root is on sys.path so src.* imports resolve
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config
from src.agents.ppo_agent import PPOAgent

print(f"Project root : {PROJECT_ROOT}")
print(f"PyTorch      : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 5)})

DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)


## 3.1 Configuration

### 3.1.1 Loading Parameters

All PPO hyperparameters are stored in `configs/ppo.yaml` and loaded via `load_config("ppo")`. This ensures that the notebook, the PPOAgent, and any future scripts all read from a single source of truth. Hardcoding hyperparameters anywhere else is deliberately avoided so that a config change propagates everywhere automatically. The key values printed below control the learning rate, clipping range, rollout length, and curriculum schedule.

In [ ]:
cfg = load_config("ppo")

print("=== PPO Hyperparameters ===")
hyper_keys = [
    "seed", "dropout", "lr", "clip_eps", "entropy_coef",
    "value_loss_coef", "n_steps", "batch_size", "n_epochs_ppo",
    "gamma", "gae_lambda", "total_timesteps",
    "curriculum_min_steps", "curriculum_perf_threshold", "max_grad_norm",
    "cam_w", "cam_h", "crop_top", "crop_bottom",
]
for k in hyper_keys:
    if k in cfg:
        print(f"  {k:35s} = {cfg[k]}")

print(f"\n=== Training Towns ===")
print(f"  {cfg['training_towns']}")

print(f"\n=== Curriculum Gate ===")
print(f"  Phase 2 unlocks when steps >= {cfg['curriculum_min_steps']}")
print(f"  AND mean episode length >= {cfg['curriculum_perf_threshold']} steps")
print(f"\n  Phase 1 weathers: {cfg['weather_phase1']}")
print(f"  Phase 2 weathers: {cfg['weather_phase2']}")

### 3.1.2 Reward Profile Comparison

Each driving style is parameterized by a weight vector

$$\mathbf{w}^{(s)} = \bigl(w_j^{(s)},\; w_s^{(s)},\; w_\delta^{(s)},\; w_l^{(s)}\bigr)$$

for $s \in \{\text{chill},\,\text{standard},\,\text{hurry}\}$, controlling the jerk, speed, steering-rate, and lane-change shaping terms respectively. A **chill** driver maximizes $w_j$ and $w_l$ to produce smooth, lane-disciplined behavior; a **hurry** driver maximizes $w_s$ while relaxing $w_j$, $w_l$, and $w_\delta$.

Importantly, **all California traffic-law penalties (collision, red lights, wrong-way, speeding, tailgating, etc.) are identical across all styles** -- only the comfort and speed-preference shaping varies with $s$.

In [ ]:
profiles = cfg["reward_profiles"]

df_profiles = pd.DataFrame(profiles).T
df_profiles.index.name = "style"
df_profiles = df_profiles[["jerk_penalty", "speed_bonus", "lane_change_penalty"]]
print(df_profiles.to_string())

## 3.2 Environment Setup

### 3.2.1 CARLA Connection Verification

Before launching any training, we verify that the CARLA simulator is reachable on `localhost:2000`. The server must already be running with the correct town loaded via the `-dx12` RHI flag, which is mandatory for the RTX 5080 Blackwell GPU to avoid camera sensor deadlocks. If the connection fails here, check that `scripts/launch_carla.bat` was executed with the correct map argument. Runtime map switching via `client.load_world()` is not used because it causes a Vulkan null-pointer crash on this hardware.

In [ ]:
try:
    import carla
    client = carla.Client("localhost", 2000)
    client.set_timeout(10.0)
    server_version = client.get_server_version()
    world = client.get_world()
    current_map = world.get_map().name
    print(f"CARLA server version : {server_version}")
    print(f"Currently loaded map : {current_map}")
except (RuntimeError, ConnectionError) as e:
    print(f"CARLA connection failed: {e}")
    print("Start CARLA first:  CarlaUE4-Win64-Shipping.exe -dx12 /Game/Carla/Maps/Town01")

### 3.2.2 BC Warm Start Verification

PPO does not train from scratch -- it starts from the BC-initialized weights produced in Notebook 02. This "warm start" gives the policy a reasonable starting behavior so that early CARLA episodes are survivable enough for PPO to collect useful reward signals. Without BC initialization, the randomly initialized policy would crash almost immediately, producing near-zero reward and making credit assignment extremely difficult. Here we verify that the BC checkpoint exists and report its parameter count.

In [ ]:
# Change SENSOR_SUITE to "multi_cam" or "lidar" to train those models.
SENSOR_SUITE = "single_cam"

bc_path = PROJECT_ROOT / "models" / f"BC_model_{SENSOR_SUITE}_best.pt"
assert bc_path.exists(), (
    f"BC checkpoint not found: {bc_path}\n"
    f"Run Notebook 02 with sensor_suite='{SENSOR_SUITE}' first."
)

bc_state_dict = torch.load(bc_path, map_location="cpu", weights_only=True)
total_params = sum(p.numel() for p in bc_state_dict.values())
print(f"BC checkpoint        : {bc_path.name}")
print(f"File size            : {bc_path.stat().st_size / 1024 / 1024:.1f} MB")
print(f"Total parameters     : {total_params:,}")
print(f"Layers               : {len(bc_state_dict)}")
print("BC weights loaded successfully -- ready for PPO warm start.")

### 3.3.1 Reward Components

The full reward signal is built from **three additive layers**:

$$r_t = \underbrace{r_t^{(1)}}_{\text{Layer 1 (CarlaEnv)}} + \underbrace{r_t^{(2)}}_{\text{Layer 2 (RoadRuleMonitor)}} + \underbrace{r_t^{(3)}}_{\text{Layer 3 (style shaping)}}$$

---

#### Layer 1 -- Base Environment (`CarlaEnv.step`) -- *identical across all styles*

| Event | $r_t^{(1)}$ | California CVC analog |
|---|---|---|
| Alive step | $+1$ | -- |
| Lane-marking invasion | $-3$ | CVC 21658 (laned roadway discipline) |
| Collision | $-200$, `terminated=True` | CVC 22350 (basic speed / negligent driving) |

#### Layer 2 -- Road Rule Monitor (`RoadRuleMonitor.step`) -- *identical across all styles*

**Tier 1** entries also set `terminated=True`:

| Infraction | $r_t^{(2)}$ | California CVC |
|---|---|---|
| Red-light violation | $-200$, terminate | CVC 21453 |
| Wrong-way driving | $-200$, terminate | CVC 21650 |
| Off-road excursion | $-200$, terminate | CVC 21663 |
| Double-solid line crossing | $-200$, terminate | CVC 21460(a) |
| Speeding above posted limit | per-step penalty | CVC 22349 / 22350 |
| Tailgating (unsafe following) | per-step penalty | CVC 21703 |
| Stop sign violation | one-time penalty | CVC 22450 |
| Solid-lane crossing | one-time penalty | CVC 21658 |
| Failure to yield | one-time penalty | CVC 21800 |

#### Layer 3 -- Style Shaping (`compute_style_reward`) -- *weights differ across styles*

Let $v_t$ be ego speed (km/h), $v_{\text{lim}}$ the posted speed limit,
$a_t = (v_t - v_{t-1})/\Delta t$ acceleration (km/h/s),
and $\delta_t \in [-1,1]$ the normalized steering action:

$$\begin{aligned}
r_t^{(3)} &= w_s \cdot \min\!\left(\frac{v_t}{v_{\text{lim}}}, 1\right) \cdot \mathbf{1}_{\text{not junction}} \\
&\quad - w_j \cdot \frac{\lvert a_t - a_{t-1}\rvert}{\Delta t \cdot 1000} \\
&\quad - w_\delta \cdot \frac{\lvert \delta_t - \delta_{t-1}\rvert}{\Delta t \cdot 10} \\
&\quad - w_l \cdot \mathbf{1}_{\text{lane changed}}
\end{aligned}$$

| Term | Formula | Purpose |
|---|---|---|
| Speed bonus | $w_s \cdot \min(v_t / v_{\text{lim}},\,1)$ | Reward speed tracking up to the posted limit (CVC 22349) |
| Jerk penalty | $w_j \cdot \lvert a_t - a_{t-1}\rvert / (\Delta t \cdot 1000)$ | Penalize abrupt acceleration (AV comfort standard) |
| Steering penalty | $w_\delta \cdot \lvert \delta_t - \delta_{t-1}\rvert / (\Delta t \cdot 10)$ | Penalize abrupt steering inputs |
| Lane-change penalty | $w_l \cdot \mathbf{1}_{\text{lane changed}}$ | Penalize unnecessary lane changes (CVC 21658) |

> **Key design choice:** all California-law penalties (Layers 1 and 2) are **mode-invariant** --
> a red-light violation costs $-200$ whether the policy is `chill` or `hurry`.
> Only the comfort and speed-preference shaping (Layer 3) varies with style.

> **Coverage gaps vs. California CVC:** the reward does not explicitly model
> pedestrian right-of-way (CVC 21950--21951), turn-signal requirements (CVC 22107--22108),
> or the basic-speed-law adverse-conditions nuance (CVC 22350).
> These are proxied by the collision penalty and CARLA NPC physics.

### 3.3.2 Style Weight Profiles

The weight vector $\mathbf{w}^{(s)} = (w_j, w_s, w_\delta, w_l)$ for each style:

| Style | $w_j$ (jerk) | $w_s$ (speed) | $w_\delta$ (steer) | $w_l$ (lane change) |
|---|---|---|---|---|
| **chill** | $2.0$ | $0.5$ | $2.0$ | $2.0$ |
| **standard** | $1.0$ | $1.0$ | $1.0$ | $1.0$ |
| **hurry** | $0.5$ | $2.0$ | $0.3$ | $0.5$ |

**Chill** down-weights the speed bonus by $4\times$ relative to **hurry** and up-weights the steering penalty by $6.7\times$, producing a conservative driver that prioritizes smoothness over throughput. **Hurry** inverts these priorities. Note that $w_l$ also varies: an in-a-hurry agent is penalized only $0.5$ per lane change vs. $2.0$ for a chill agent -- though CVC 21658 makes no such distinction. All weights are loaded from `configs/ppo.yaml` and passed to `compute_style_reward()` at every environment step.

## 3.4 Training

### 3.4.1 Autonomous Multi-Town Training

PPO alternates between rollout collection and policy update. At each update, a buffer of $N = 512$ transitions $(s_t, a_t, r_t, s_{t+1})$ is collected and the Generalized Advantage Estimate

$$\hat{A}_t = \sum_{k=0}^{T-t}(\gamma\lambda)^k\,\delta_{t+k}, \qquad \delta_t = r_t + \gamma V_\phi(s_{t+1}) - V_\phi(s_t)$$

is computed with $\gamma = 0.99$ and $\lambda = 0.95$. The buffer is split into mini-batches of size $64$ and the network is updated for $K = 4$ epochs per rollout. The combined objective is

$$\mathcal{L}(\theta, \phi) = \mathcal{L}^{\text{CLIP}}(\theta) - c_v\,\mathcal{L}^{\text{VF}}(\phi) + c_H\,H[\pi_\theta], \quad c_v = 0.5,\; c_H = 0.01$$

The agent launches a fresh CARLA process for each town, collects one rollout batch, updates the model, then moves to the next town. You do **not** need to start or stop CARLA manually. Each style takes roughly 8--10 hours to reach $200{,}000$ total timesteps on an RTX 5080.

In [ ]:
agent_chill = PPOAgent(
    bc_checkpoint=str(PROJECT_ROOT / "models" / f"BC_model_{SENSOR_SUITE}_best.pt"),
    style="chill",
    sensor_suite=SENSOR_SUITE,
    save_dir=str(MODELS_DIR),
    results_dir=str(RESULTS_DIR),
)
try:
    history_chill = agent_chill.run()
finally:
    # Guarantees env.close() fires even if the cell errors, so CARLA is not
    # left in synchronous mode for the next cell or a kernel restart.
    if hasattr(agent_chill, "_env") and agent_chill._env is not None:
        agent_chill._env.close()
print("Chill training complete.")

## 3.5 Training -- Standard Style

**Weight vector:** $(w_j,\,w_s,\,w_\delta,\,w_l) = (1.0,\,1.0,\,1.0,\,1.0)$. All shaping terms contribute equally; this is the neutral reference profile. Do **not** start CARLA manually -- the agent manages the process lifecycle automatically.

In [ ]:
agent_standard = PPOAgent(
    bc_checkpoint=str(PROJECT_ROOT / "models" / f"BC_model_{SENSOR_SUITE}_best.pt"),
    style="standard",
    sensor_suite=SENSOR_SUITE,
    save_dir=str(MODELS_DIR),
    results_dir=str(RESULTS_DIR),
)
try:
    history_standard = agent_standard.run()
finally:
    if hasattr(agent_standard, "_env") and agent_standard._env is not None:
        agent_standard._env.close()
print("Standard training complete.")

## 3.6 Training -- Hurry Style

**Weight vector:** $(w_j,\,w_s,\,w_\delta,\,w_l) = (0.5,\,2.0,\,0.3,\,0.5)$. The speed bonus is $4\times$ the chill value; jerk and lane-change penalties are halved; the steering penalty is reduced by $6.7\times$. This produces an aggressive driver that prioritizes speed over comfort. Do **not** start CARLA manually -- the agent manages the process lifecycle automatically.

In [ ]:
agent_hurry = PPOAgent(
    bc_checkpoint=str(PROJECT_ROOT / "models" / f"BC_model_{SENSOR_SUITE}_best.pt"),
    style="hurry",
    sensor_suite=SENSOR_SUITE,
    save_dir=str(MODELS_DIR),
    results_dir=str(RESULTS_DIR),
)
try:
    history_hurry = agent_hurry.run()
finally:
    if hasattr(agent_hurry, "_env") and agent_hurry._env is not None:
        agent_hurry._env.close()
print("Hurry training complete.")

### 3.7.1 Policy Loss Curves

The plotted quantity is the PPO clipped surrogate loss

$$\mathcal{L}^{\text{CLIP}} = \mathbb{E}_t\!\left[\min\!\left(\rho_t\,\hat{A}_t,\;\operatorname{clip}(\rho_t,\,1{-}\varepsilon,\,1{+}\varepsilon)\,\hat{A}_t\right)\right], \quad \rho_t = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$$

A healthy run shows the loss starting negative (policy improving over BC initialization) and stabilizing near zero as the policy converges. Large spikes may indicate the curriculum transition where the weather pool expands from $\{\text{ClearNoon}\}$ to all six presets, forcing the policy to generalize.

In [ ]:
STYLES = ["chill", "standard", "hurry"]
STYLE_COLORS = {"chill": "tab:blue", "standard": "tab:green", "hurry": "tab:orange"}

fig, ax = plt.subplots(figsize=(10, 5))

for style in STYLES:
    hist_path = RESULTS_DIR / f"ppo_{style}_{SENSOR_SUITE}_training_history.json"
    if hist_path.exists():
        with open(hist_path) as f:
            hist = json.load(f)
        ax.plot(hist["policy_losses"], label=style, color=STYLE_COLORS[style], alpha=0.8)

ax.set_ylabel("Policy Loss")
ax.set_title(f"Policy Loss Curves -- All Styles ({SENSOR_SUITE})")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 3.7.2 Value Loss Curves

The critic minimizes the clipped mean-squared TD error

$$\mathcal{L}^{\text{VF}}(\phi) = \mathbb{E}_t\!\left[\bigl(V_\phi(s_t) - \hat{V}_t^{\text{target}}\bigr)^2\right], \quad \hat{V}_t^{\text{target}} = \hat{A}_t + V_{\phi_{\text{old}}}(s_t)$$

Because the **hurry** style's speed bonus ($w_s = 2.0$) inflates absolute reward magnitudes, its value loss will be larger than **chill** ($w_s = 0.5$) in absolute terms for the same physical trajectory -- the two are not directly comparable. What matters is the downward trend *within* each style.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for style in STYLES:
    hist_path = RESULTS_DIR / f"ppo_{style}_{SENSOR_SUITE}_training_history.json"
    if hist_path.exists():
        with open(hist_path) as f:
            hist = json.load(f)
        ax.plot(hist["value_losses"], label=style, color=STYLE_COLORS[style], alpha=0.8)

ax.set_ylabel("Value Loss")
ax.set_title(f"Value Loss Curves -- All Styles ({SENSOR_SUITE})")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 3.7.3 Episode Reward Curves

Episode reward is the total shaped return $R = \sum_{t=0}^{T} r_t$ accumulated in a CARLA episode. An upward trend indicates the policy is learning to drive better under its style-specific objective. Note that raw reward magnitudes are **not comparable across styles**: the hurry speed bonus weight $w_s = 2.0$ inflates $R$ by up to $4\times$ relative to chill ($w_s = 0.5$) for the same physical trajectory.

The curriculum transition fires once at step $t \geq 50{,}000$ when the mean episode length exceeds $800$ steps. A transient performance dip is expected as the policy encounters novel lighting and precipitation conditions for the first time.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for style in STYLES:
    hist_path = RESULTS_DIR / f"ppo_{style}_{SENSOR_SUITE}_training_history.json"
    if hist_path.exists():
        with open(hist_path) as f:
            hist = json.load(f)
        rewards = hist["episode_rewards"]
        ax.plot(rewards, label=style, color=STYLE_COLORS[style], alpha=0.3, linewidth=0.8)
        if len(rewards) > 10:
            rolling = np.convolve(rewards, np.ones(10) / 10, mode="valid")
            ax.plot(range(9, 9 + len(rolling)), rolling,
                    color=STYLE_COLORS[style], linewidth=2, alpha=0.9)

ax.set_ylabel("Episode Reward")
ax.set_title(f"Episode Reward Curves -- All Styles ({SENSOR_SUITE})")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 3.7.4 Entropy Curves

For the diagonal-Gaussian action distribution $\pi_\theta(\cdot|s) = \mathcal{N}(\mu_\theta(s),\,\sigma^2 I)$, the differential entropy is

$$H[\pi_\theta] = \frac{d}{2}\ln(2\pi e\,\sigma^2)$$

where $d = 2$ is the action dimension (steering, throttle/brake). The entropy bonus $c_H H[\pi_\theta]$ (with $c_H = 0.01$) is added to the PPO objective to prevent premature convergence to a deterministic policy. Entropy should decrease gradually as the policy specializes but should not collapse to zero: a fully deterministic policy ($\sigma \to 0$) cannot adapt to novel scenarios. If entropy collapses early, increase `entropy_coef` in `configs/ppo.yaml`.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for style in STYLES:
    hist_path = RESULTS_DIR / f"ppo_{style}_{SENSOR_SUITE}_training_history.json"
    if hist_path.exists():
        with open(hist_path) as f:
            hist = json.load(f)
        ax.plot(hist["entropy_bonuses"], label=style, color=STYLE_COLORS[style], alpha=0.8)

ax.set_ylabel("Entropy")
ax.set_title(f"Entropy Curves -- All Styles ({SENSOR_SUITE})")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3.8 Save

### 3.8.1 Checkpoint Inventory

After all training runs complete, we verify that the expected checkpoint files exist on disk. Each town-style combination should produce one `ppo_{town}_{style}_best.pt` file. Missing checkpoints indicate that training did not reach the minimum reward threshold for that combination, or that CARLA crashed during training. The file sizes are printed as a sanity check -- all checkpoints should be roughly the same size since they share the same ActorCritic architecture.

In [ ]:
print(f"=== PPO Checkpoint Inventory ({SENSOR_SUITE}) ===")
print(f"{'Checkpoint':<50s} {'Size (MB)':>10s} {'Status':>8s}")
print("-" * 70)

found = 0
for style in STYLES:
    ckpt = MODELS_DIR / f"ppo_{style}_{SENSOR_SUITE}_best.pt"
    if ckpt.exists():
        size_mb = ckpt.stat().st_size / 1024 / 1024
        print(f"{ckpt.name:<50s} {size_mb:>9.1f}M {'OK':>8s}")
        found += 1
    else:
        print(f"{ckpt.name:<50s} {'---':>10s} {'MISSING':>8s}")

print(f"\nFound {found}/{len(STYLES)} checkpoints for {SENSOR_SUITE}.")
print("Re-run NB03 with sensor_suite='multi_cam' and 'lidar' for the other two suites.")

### 3.8.2 Config Snapshot

We save a JSON snapshot of the PPO configuration that was active during this notebook run. This creates a permanent record alongside the training histories so that future analysis can always trace which hyperparameters produced each set of checkpoints. The snapshot is written to `results/ppo_config.json` and includes all values from `configs/ppo.yaml` plus the reward profiles.

In [ ]:
config_snapshot_path = RESULTS_DIR / "ppo_config.json"
config_snapshot_path.parent.mkdir(parents=True, exist_ok=True)

with open(config_snapshot_path, "w") as f:
    json.dump(cfg, f, indent=2)

print(f"Config snapshot saved to: {config_snapshot_path}")
print(f"File size: {config_snapshot_path.stat().st_size} bytes")
print("\nNotebook 03 complete. Proceed to Notebook 04 for evaluation.")